In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

import medicalDataLoader_Enes as medicalDataLoader
from progressBar import printProgressBar
from utils_Enes import (to_var, getTargetSegmentation, validate, show_predictions, DiceLoss, show_confusion,
                        make_pseudo_labels, show_pseudo_labels)
from UNet_Base_Enes import UNet

In [ ]:
TEACHER_NAME = 'final_dice'
MODEL_NAME = 'pseudo_labels'
EPOCHS = 100
BATCH_SIZE = 8
LR = 0.001
AUGMENT = True
AUG_FLIP = False
AUG_SCALE = True
AUG_INTENSITY = True
SCHEDULER = True
WEIGHT_DECAY = 0
LOSS = 'ce+dice'
SEED = 0
THRESHOLD = 0.95
PSEUDO_WEIGHT = 0.5
RAMPUP_EPOCHS = 10
STUDENT_EPOCHS = 50
STUDENT_LR = 0.0001

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/datasets/enesozkosar/acdc-heartbeat/Data/'
else:
    DATA_DIR = '../../Data/'

In [ ]:
def runTraining(model_name, pseudo_set=None, init=None, epochs=EPOCHS, lr=LR):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    print('Dataset :', DATA_DIR)
    transform = transforms.Compose([transforms.ToTensor()])

    train_set = medicalDataLoader.MedicalImageDataset('train', DATA_DIR, transform=transform,
                                                      mask_transform=transform, augment=AUGMENT,
                                                      flip=AUG_FLIP, scale=AUG_SCALE, intensity=AUG_INTENSITY)
    train_eval_set = medicalDataLoader.MedicalImageDataset('train', DATA_DIR, transform=transform,
                                                           mask_transform=transform)
    val_set = medicalDataLoader.MedicalImageDataset('val', DATA_DIR, transform=transform,
                                                    mask_transform=transform)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    train_eval_loader = DataLoader(train_eval_set, batch_size=BATCH_SIZE, shuffle=False)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

    pseudo_loader = None
    if pseudo_set is not None:
        pseudo_loader = DataLoader(pseudo_set, batch_size=BATCH_SIZE, shuffle=True)
        pseudo_iter = iter(pseudo_loader)

    net = UNet(num_classes=4)
    if init is not None:
        net.load_state_dict(init.state_dict())
    if torch.cuda.is_available():
        net.cuda()
    print('Paramètres : {:,}'.format(sum(p.numel() for p in net.parameters())))

    CE_loss = nn.CrossEntropyLoss(ignore_index=255)
    dice_loss = DiceLoss()

    def loss_fn(out, target):
        if LOSS == 'ce+dice':
            return CE_loss(out, target) + dice_loss(out, target)
        return CE_loss(out, target)

    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = None
    if SCHEDULER:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    model_dir = os.path.join('models', model_name)
    os.makedirs(model_dir, exist_ok=True)
    best_path = os.path.join(model_dir, 'best.pth')
    history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': [], 'val_hd': []}
    best_dsc, best_epoch = -1, 0

    for epoch in range(epochs):
        net.train()
        weight = PSEUDO_WEIGHT * min(1.0, (epoch + 1) / RAMPUP_EPOCHS)
        losses = []
        for j, (images, labels, _) in enumerate(train_loader):
            images = to_var(images)
            target = getTargetSegmentation(to_var(labels))

            optimizer.zero_grad()
            loss = loss_fn(net(images), target)

            if pseudo_loader is not None:
                try:
                    u_images, u_target, _ = next(pseudo_iter)
                except StopIteration:
                    pseudo_iter = iter(pseudo_loader)
                    u_images, u_target, _ = next(pseudo_iter)
                u_loss = loss_fn(net(to_var(u_images)), to_var(u_target))
                loss = loss + weight * u_loss

            loss.backward()
            optimizer.step()

            losses.append(loss.item())
            printProgressBar(j + 1, len(train_loader), prefix='[Training] Epoch: {} '.format(epoch),
                             length=15, suffix=' Loss: {:.4f}'.format(loss.item()))
        train_loss = np.mean(losses)
        current_lr = optimizer.param_groups[0]['lr']
        if scheduler is not None:
            scheduler.step()

        _, train_dsc, _ = validate(net, train_eval_loader, loss_fn, with_hausdorff=False)
        val_loss, val_dsc, val_hd = validate(net, val_loader, loss_fn)
        mean_dsc = np.mean(val_dsc)
        print('Epoch {:2d} | lr {:.6f} | loss train {:.4f}  val {:.4f}'.format(epoch, current_lr, train_loss, val_loss))
        if pseudo_loader is not None:
            print('   Poids pseudo-labels {:.2f}'.format(weight))
        print('   Train Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*train_dsc, np.mean(train_dsc)))
        print('   Val   Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*val_dsc, mean_dsc))
        print('   Val   Hausdorff (px)  VD {:.1f}  Myo {:.1f}  VG {:.1f}'.format(*val_hd))

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dsc)
        history['val_dice'].append(val_dsc)
        history['val_hd'].append(val_hd)

        if mean_dsc > best_dsc:
            best_dsc, best_epoch = mean_dsc, epoch
            torch.save(net.state_dict(), best_path)
            print('   -> meilleur modèle enregistré')

    np.save(os.path.join(model_dir, 'history.npy'), history)
    print('Meilleur Dice moyen : {:.4f} (époque {})'.format(best_dsc, best_epoch))
    last10 = np.mean([np.mean(d) for d in history['val_dice'][-10:]])
    print('Dice moyen des 10 dernières époques : {:.4f}'.format(last10))

    net.load_state_dict(torch.load(best_path))
    return net, train_eval_loader, val_loader

In [ ]:
teacher_path = os.path.join('..', 'baseline', 'models', TEACHER_NAME, 'best.pth')
if os.path.exists(teacher_path):
    teacher = UNet(num_classes=4)
    teacher.load_state_dict(torch.load(teacher_path, map_location='cpu'))
    if torch.cuda.is_available():
        teacher.cuda()
    print('Modèle 1A chargé :', teacher_path)
else:
    teacher, _, _ = runTraining(TEACHER_NAME)

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
unlabeled_set = medicalDataLoader.UnlabeledDataset(DATA_DIR, transform=transform)
unlabeled_loader = DataLoader(unlabeled_set, batch_size=BATCH_SIZE, shuffle=False)
pseudo_masks = make_pseudo_labels(teacher, unlabeled_loader, THRESHOLD)

heart = (pseudo_masks > 0) & (pseudo_masks < 255)
print('Images non annotées : {}'.format(len(pseudo_masks)))
print('Pixels ignorés (confiance < {}) : {:.2f} %'.format(THRESHOLD, 100 * (pseudo_masks == 255).mean()))
print('Images avec du cœur dans le pseudo-label : {} / {}'.format(heart.any(axis=(1, 2)).sum(), len(pseudo_masks)))
show_pseudo_labels(unlabeled_set, pseudo_masks, [0, 250, 500, 750, 1000])

pseudo_set = medicalDataLoader.PseudoLabelDataset(DATA_DIR, pseudo_masks, transform=transform, augment=AUGMENT,
                                                  flip=AUG_FLIP, scale=AUG_SCALE, intensity=AUG_INTENSITY)

In [ ]:
net, train_loader, val_loader = runTraining(MODEL_NAME, pseudo_set, init=teacher, epochs=STUDENT_EPOCHS, lr=STUDENT_LR)

CE_loss = nn.CrossEntropyLoss()
for name, model in [('1A (supervisé)', teacher), ('1B (pseudo-labels)', net)]:
    for post in [False, True]:
        _, dsc, hd = validate(model, val_loader, CE_loss, post_process=post)
        print('{} {} post-traitement'.format(name, 'avec' if post else 'sans'))
        print('   Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*dsc, np.mean(dsc)))
        print('   Hausdorff (px)  VD {:.1f}  Myo {:.1f}  VG {:.1f}'.format(*hd))

show_predictions(net, val_loader, title='Validation 1B avec post-traitement', post_process=True)
show_confusion(net, val_loader, title='(1B, avec post-traitement)', post_process=True)